# Gemini OCR 

In [1]:

import os, base64, time, glob
import pandas as pd
import fitz                       # PyMuPDF 
from openai import OpenAI         # used in nb05


GEMINI_KEY = ""         

GEMINI_MODEL   = "gemini-2.5-flash-lite"
CACHE_DIR      = "attachment_cache_fixed"          # nb02's PDF download folder
NEEDS_OCR_CSV  = "still_need_ocr.csv"              # nb02 output. 
OUT_FILE_CSV   = "attachment_ocr_text.csv"         
OUT_DOC_CSV    = "attachment_ocr_per_comment.csv"  # combined per Document ID
MAIN_CSV       = "cleaned_comments_ver2.csv"
OUT_MAIN_CSV   = "cleaned_comments_ver2_with_ocr.csv"  
MAX_PAGES      = 15        
DPI            = 200       

client = OpenAI(api_key=GEMINI_KEY,
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                timeout=120, max_retries=0)

OCR_PROMPT = ("You are an OCR system. Transcribe ALL text in this page image exactly as "
              "written, including handwriting, letterhead and signatures. Preserve line "
              "breaks. Return ONLY the transcribed text, no commentary. If the page is "
              "blank or fully illegible, return an empty string.")

def doc_id_from_name(fn):
   
    
    b = os.path.basename(str(fn))
    return b.split("__")[0] if "__" in b else b

print("setup OK — key set:", GEMINI_KEY != "")

setup OK — key set: False


In [2]:

from tqdm.auto import tqdm

if NEEDS_OCR_CSV and os.path.exists(NEEDS_OCR_CSV):
    todo = pd.read_csv(NEEDS_OCR_CSV)
    col = "local_path" if "local_path" in todo.columns else todo.columns[-1]
    paths = [p for p in todo[col].astype(str) if p.lower().endswith(".pdf")]
else:
    paths = glob.glob(os.path.join(CACHE_DIR, "*.pdf"))
paths = [p for p in paths if os.path.exists(p)]

prev = pd.read_csv(OUT_FILE_CSV) if os.path.exists(OUT_FILE_CSV) else None
done = set(prev["local_path"].astype(str)) if prev is not None else set()
todo_paths = [p for p in paths if p not in done]
print(f"total PDFs: {len(paths)} | already done: {len(done)} | to do now: {len(todo_paths)}")

def ocr_page(png_bytes):
    b64 = base64.b64encode(png_bytes).decode()
    r = client.chat.completions.create(
        model=GEMINI_MODEL, temperature=0,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": OCR_PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]}])
    return (r.choices[0].message.content or "").strip()

def ocr_pdf(path):
    parts = []
    doc = fitz.open(path)
    for i, page in enumerate(doc):
        if i >= MAX_PAGES:
            break
        png = page.get_pixmap(dpi=DPI).tobytes("png")
        for k in range(3):                       
            try:
                parts.append(ocr_page(png)); break
            except Exception:
                if k == 2: raise
                time.sleep(min(2 ** k, 8))
    doc.close()
    return "\n\n".join(p for p in parts if p)

recs = prev.to_dict("records") if prev is not None else []
new, failed = 0, 0
pbar = tqdm(todo_paths, desc="OCR", unit="pdf")
for path in pbar:
    rec = {"Document ID": doc_id_from_name(path), "local_path": path,
           "filename": os.path.basename(path), "ocr_text": "", "ocr_failed": False, "err": ""}
    try:
        rec["ocr_text"] = ocr_pdf(path)
    except Exception as e:
        rec["ocr_failed"] = True; rec["err"] = str(e)[:200]
    failed += int(rec["ocr_failed"])
    rec["ocr_char_len"] = len(rec["ocr_text"])
    recs.append(rec); new += 1
    pbar.set_postfix(done=len(recs), failed=failed, chars=rec["ocr_char_len"])  
    if new % 10 == 0:
        pd.DataFrame(recs).to_csv(OUT_FILE_CSV, index=False)                   
    time.sleep(0.3)

pd.DataFrame(recs).to_csv(OUT_FILE_CSV, index=False)
print(f"OCR done -> {OUT_FILE_CSV}: {len(recs)} rows, {failed} failed")

total PDFs: 223 | already done: 413 | to do now: 0


OCR: 0pdf [00:00, ?pdf/s]

OCR done -> attachment_ocr_text.csv: 413 rows, 0 failed


In [3]:

ocr = pd.read_csv(OUT_FILE_CSV)
ocr["ocr_text"] = ocr["ocr_text"].fillna("").astype(str)
per_doc = (ocr.groupby("Document ID")
              .agg(ocr_text=("ocr_text", lambda s: "\n\n".join(x for x in s if x and x != "nan")),
                   n_ocr_files=("filename", "count"))
              .reset_index())
per_doc["ocr_char_len"] = per_doc["ocr_text"].str.len()
per_doc.to_csv(OUT_DOC_CSV, index=False)
print(f"per-comment -> {OUT_DOC_CSV}: {len(per_doc)} comments, "
      f"{int((per_doc['ocr_char_len'] > 0).sum())} with usable text")

main = pd.read_csv(MAIN_CSV, low_memory=False)
main = main.merge(per_doc[["Document ID", "ocr_text"]], on="Document ID", how="left")
# combined text = inline comment + OCR'd attachment text (this is what you re-label on)
main["llm_input_text"] = (main["comment_text"].fillna("").astype(str) + "\n\n" +
                          main["ocr_text"].fillna("").astype(str)).str.strip()
main.to_csv(OUT_MAIN_CSV, index=False)
print(f"merged -> {OUT_MAIN_CSV}: {int(main['ocr_text'].notna().sum())} comments now carry OCR text")
print("\nNEXT: re-run nb05 labelling on the OCR'd rows using TEXT_COL = 'llm_input_text'.")

per-comment -> attachment_ocr_per_comment.csv: 258 comments, 178 with usable text
merged -> cleaned_comments_ver2_with_ocr.csv: 258 comments now carry OCR text

NEXT: re-run nb05 labelling on the OCR'd rows using TEXT_COL = 'llm_input_text'.
